In [ ]:
import scanpy as sc
import rpy2.robjects as ro
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import os
import harmonypy as hm
from scipy.stats import median_abs_deviation as mad
from pathlib import Path
from src import *

%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

sc.set_figure_params(dpi=150, facecolor='white')
sc.settings.verbosity = 3

project_root = Path.cwd().parent
data_dir = project_root / "data"
results_dir = project_root / "results"
ro.r(f'renv::load("{project_root}")')

In [ ]:
adatas = assign_adatas(data_dir / "raw" / "filtered_barcode_feature_matrices")

In [ ]:
%%R
library(SoupX)

make_soup <- function(data, raw, genes, cells, soupx_groups, seed=1) {
    set.seed(seed)
    rownames(data) <- genes
    colnames(data) <- cells
    rownames(raw) <- genes
    data <- as(data, "sparseMatrix")
    raw <- as(raw, "sparseMatrix")
    sc <- SoupChannel(raw, data, calcSoupProfile = TRUE)
    sc <- setClusters(sc, soupx_groups)
    sc <- autoEstCont(sc, doPlot = FALSE)
    adjustCounts(sc, roundToInt = TRUE)
}

In [ ]:
raw_dir = data_dir / "raw" / "raw_barcode_feature_matrices"
adatas = [cook_soup(a, raw_dir) for a in adatas]

In [ ]:
print("raw total:", adatas[0].layers["raw_counts"].sum())
print("decontaminated total:", adatas[0].layers["soupX_counts"].sum())
print("X == soupX:", (adatas[0].X != adatas[0].layers["soupX_counts"]).nnz == 0)

In [ ]:
for i, adata in enumerate(adatas):
    sample = adata.obs["sample_id"].iloc[0]
    sex = adata.obs["sex"].iloc[0]
    genotype = adata.obs["genotype"].iloc[0]

    out_dir = data_dir / "processed" / "decontaminated_matrices"
    decont = os.path.join(out_dir, f"{sample}_{sex}_{genotype}_decontaminated.h5")
    adata.write_h5ad(decont)

In [ ]:
%%R
library(scDblFinder)
library(SingleCellExperiment)

find_doublets <- function(counts, genes, cells, seed=1) {
    set.seed(seed)
    rownames(counts) <- genes
    colnames(counts) <- cells
    counts <- as(counts, "sparseMatrix")
    sce <- scDblFinder(SingleCellExperiment(assays = list(counts = counts)))
    data.frame(score = sce$scDblFinder.score,
               class = as.character(sce$scDblFinder.class),
               row.names = colnames(sce))
}

In [ ]:
adatas = [flag_doublets(a) for a in adatas]

In [ ]:
adatas = [calculate_qc_metrics(a) for a in adatas]
adatas = [flag_outliers(a) for a in adatas]

In [ ]:
plot_qc_metrics(adatas, hue="outlier", palette="husl")

In [ ]:
print("Remaining cells before filtering outliers:")
for a in adatas:
    print(f" {a.obs['sample_id'].iloc[0]}: {a.n_obs} cells")
adatas = [filter_outliers(a) for a in adatas]
print("Remaining cells after filtering outliers:")
for a in adatas:
    print(f" {a.obs['sample_id'].iloc[0]}: {a.n_obs} cells")

In [ ]:
adata = sc.concat(adatas, join="inner", merge="same")
adata.write_h5ad(data_dir / "processed" / "00_combined_rawcounts.h5ad")
del adatas

In [ ]:
adata = deviant_hvg(adata, n_top=4000, layer="soupX_counts")
plot_deviance(adata, figsize=(12, 4))

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.pca(adata, n_comps=50, use_highly_variable=True, svd_solver="arpack")
sc.pl.pca_overview(adata, color="sample_id")

In [ ]:
sc.pp.neighbors(adata, n_pcs=20, random_state=1)
sc.tl.umap(adata, maxiter=500, random_state=1)

In [ ]:
sc.tl.leiden(adata, resolution=0.35, flavor="igraph", n_iterations=2, random_state=1)

In [ ]:
sc.pl.umap(adata,
           color=["leiden", "sample_id", "log1p_total_counts", "scDblFinder_score"],
           ncols=2)
sc.pl.umap(adata,
           color=["Ptprc", "Epcam", "Cdh5", "Col6a3"],
           vmin=0, vmax="p99", ncols=2)

In [ ]:
print("Removing residual doublets; number of cells before:", adata.n_obs)
adata = adata[adata.obs["leiden"] != "0"].copy()
print("Number of cells after:", adata.n_obs)

In [ ]:
sc.pp.pca(adata, n_comps=50, use_highly_variable=True, svd_solver="arpack")
sc.pl.pca_overview(adata, color="sample_id")

In [ ]:
ho = hm.run_harmony(
    adata.obsm["X_pca"],
    adata.obs,
    vars_use=["sample_id"]
)
adata.obsm["X_pca_harmony"] = ho.Z_corr

In [ ]:
sc.pp.neighbors(adata, n_pcs=20, use_rep="X_pca_harmony", random_state=1)
sc.tl.umap(adata, maxiter=500, random_state=1)

In [ ]:
sc.tl.leiden(adata, resolution=0.75, flavor="igraph", n_iterations=2, random_state=1)

In [ ]:
sc.pl.umap(adata,
           color=["leiden", "pct_counts_mt", "log1p_total_counts", "scDblFinder_score"],
           ncols=2, legend_loc="on data")
sc.pl.umap(adata,
           color=["Ptprc", "Epcam", "Cdh5", "Col6a3"],
           vmin=0, vmax="p99", ncols=2)

In [ ]:
lineage_labels = {
    "immune": ["2", "21", "22", "23", "1"],
    "endothelial": ["4", "5", "6", "24", "20", "18"],
    "epithelial": ["7", "17", "15", "8", "0"],
    "mesenchymal": ["9", "12", "11", "13", "14", "19", "10", "16", "3"]
}

apply_celltypes(adata, lineage_labels, key="leiden", out="lineage")

In [ ]:
sc.pl.umap(adata, color="lineage")

In [ ]:
adata.write_h5ad(data_dir / "processed" / "00_combined_processed.h5ad")